# Stage 2: Coding the Attention mechanism

Before jumping into code a full multi-head attention, we'll go step by step.

## 1. Simple Self-attention

Without parameters

In [1]:
import torch

inputs = torch.tensor([
    [0.43, 0.15, 0.89],  # Your
    [0.55, 0.87, 0.66],  # journey
    [0.57, 0.85, 0.64],  # starts
    [0.22, 0.58, 0.33],  # with
    [0.77, 0.25, 0.10],  # one
    [0.05, 0.80, 0.55],  # step
])

print(inputs.shape)
# (T, C) = (6, 3)

torch.Size([6, 3])


In [2]:
query = inputs[1]

In [3]:
attention_scores = torch.empty(inputs.shape[0])

for i, token_embedding in enumerate(inputs):
    attention_scores[i] = torch.dot(
        query,
        token_embedding,
    )

print(attention_scores)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


Now, we apply softmax to normalize. It's better than regular normalization as it handles well negative input an sum to zero thanks to euclidean number

In [4]:
attention_weights = torch.softmax(
    attention_scores,
    dim=0
)

print(attention_weights)
print(attention_weights.sum())

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
tensor(1.)


Now, obtain the context vector

In [ ]:
context_vector = torch.zeros(query.shape)

for i, token_embedding in enumerate(inputs):
    context_vector += (
        attention_weights[i] * token_embedding
    )

print(context_vector)
print(context_vector.shape)

tensor([0.4419, 0.6515, 0.5683])
torch.Size([3])


Now, we'll obtain the context vector for each input token

In [6]:
attention_scores = inputs @ inputs.T

print(attention_scores)
print(attention_scores.shape)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
torch.Size([6, 6])


In [12]:
attention_weights = torch.softmax(
    attention_scores,
    dim=-1
)

print(attention_weights)
print(attention_weights.shape)
print(attention_weights.sum(dim=-1))

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
torch.Size([6, 6])
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [15]:
context_vectors = attention_weights @ inputs

print(context_vectors)
print(context_vectors.shape)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])
torch.Size([6, 3])


## 2. Add trainable parameters Q, K, V

In [ ]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

In [19]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(
    torch.rand(d_in, d_out),
    requires_grad=False
)

W_key = torch.nn.Parameter(
    torch.rand(d_in, d_out),
    requires_grad=False
)

W_value = torch.nn.Parameter(
    torch.rand(d_in, d_out),
    requires_grad=False
)

print(W_query)

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]])


In [20]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

print("Original:", x_2, x_2.shape)
print("Query:", query_2, query_2.shape)
print("Key:", key_2, key_2.shape)
print("Value:", value_2, value_2.shape)

Original: tensor([0.5500, 0.8700, 0.6600]) torch.Size([3])
Query: tensor([0.4306, 1.4551]) torch.Size([2])
Key: tensor([0.4433, 1.1419]) torch.Size([2])
Value: tensor([0.3951, 1.0037]) torch.Size([2])


Now we transform all input tokens

In [21]:
queries = inputs @ W_query
keys = inputs @ W_key
values = inputs @ W_value

print("Queries:", queries.shape)
print("Keys:", keys.shape)
print("Values:", values.shape)

Queries: torch.Size([6, 2])
Keys: torch.Size([6, 2])
Values: torch.Size([6, 2])


Query: what the token wants to know

Key: how the token can be found

Value: what information does the token provide∫

In [ ]:
attention_scores_2 = query_2 @ keys.T
# how much the token 2 must look at other tokens

print(attention_scores_2)
print(attention_scores_2.shape)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])
torch.Size([6])


Now, we need to scale the scores and apply softmax to obtain attention weights

attention weights = softmax(QKᵀ / √dₖ)

In [29]:
import math

attention_weights_2 = torch.softmax(
    attention_scores_2 / math.sqrt(d_out),
    dim = -1
)

print(attention_weights_2)
print(attention_weights_2.sum())

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])
tensor(1.)


We need to scale because with larger scores, softmax tends to saturate

Now that we have how much each token looks at each other, we need to take the information they can provide --> CONTEXT VECTOR

In [34]:
context_vector_2 = attention_weights_2 @ values

print(context_vector_2)
print(context_vector_2.shape)

tensor([0.3061, 0.8210])
torch.Size([2])


At this point, we already 